<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_106_mlp_probing_steering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧬 **Module 106 — MLP Probing + Steering** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 15 · Mechanistic Interpretability Workshop · **FINAL MODULE 106/106**


# Module 106 — MLP Probing + Steering

> **The final module of the course.** Cohen Ch7 (projects 42-50) finishes the mech-interp workshop with the part most often skipped: the **MLP / FFN block**, which holds **two-thirds of a Transformer's parameters** but is dramatically less studied than attention. You'll dissect MLP weights and activations, find **grammar neurons** and **truth-conditional neurons**, measure **Minkowski distances** and **mutual information**, run **statistical lesioning**, build **XGBoost probes** and **logistic-regression "can / can't" classifiers** on hidden states, ship the canonical **median-replacement** steering intervention, and turn the MLP's projection layer into a **recommender system**. By the end you'll have the full mech-interp pipeline — from input token statistics (M101) through to MLP steering (this module).

### What you'll cover
1. The **MLP block** revisited — why it matters
2. **MLP weights + activations characteristics**
3. **Characterizing the MLP progression** across layers
4. **Grammar neurons** — does the network learn POS?
5. **Minkowski distance + mutual information** + token positions
6. **Statistics-based lesioning** of MLP neurons
7. **Supervised probing with XGBoost**
8. **"Can" vs "can't" logistic-regression probe** — capability detection
9. **Median-replacement steering** of MLP activations
10. **Recommender systems with MLP projections** + the 106-module epilogue


## 1 · The MLP block revisited

A standard Transformer decoder block has two sub-modules:

```
h_in
 │
 ▼ LayerNorm
 ▼ Multi-Head Attention   ──►   write to residual  (M105)
 │
 ▼ LayerNorm
 ▼ MLP / FFN              ──►   write to residual  (this module)
 │
h_out
```

The MLP (a.k.a. FFN, a.k.a. *the perceptron block*) is:

```
h ∈ ℝ^D
    │
    ▼ W_up   ∈ ℝ^{D_FF × D}     D_FF = 4·D (GPT-2, BERT)  or  ~3.5·D (Llama-3 with SwiGLU)
    │
    ▼ activation  σ                            (ReLU, GELU, SwiGLU)
    │
    ▼ W_down ∈ ℝ^{D × D_FF}
    │
out ∈ ℝ^D
```

**Two-thirds of a Transformer's parameters are in `W_up · W_down`** — yet for years mech-interp focused on attention. The MLP is "where the model stores facts" (Anthropic 2022 "Transformer Circuits Thread"; **ROME / MEMIT** edit MLPs to update facts in 2023).

**SwiGLU variant (Llama, Mistral, Qwen)**: replaces the single up-projection with `(W_up · h) ⊙ swish(W_gate · h)`, doubling the parameter count of the up-projection but improving quality. This is the 2026 production default.

> 🧠 **The MLP-as-key-value-store view (Geva 2021, 2022).** Treat the rows of `W_up` as **keys** that detect features in `h`; the rows of `W_down` as **values** that get written back to the residual stream when the corresponding key fires. Then an MLP layer is literally a *content-addressable memory* — explaining why factual recall lives in MLP and why ROME (2023) edits factual associations by surgically modifying `W_down` rows.


## 2 · MLP weights + activations

Cohen proj 42 analyzes the weight matrices:

| Stat | Tells you |
|---|---|
| **`||W_up.row||`** distribution | Which "keys" are strong / weak — many rows have near-zero norm (the dead-neuron problem) |
| **`||W_down.col||`** distribution | How much each neuron writes back to residual |
| **Cosine(W_up.row_i, W_up.row_j)** | Correlated keys = overlapping feature detectors |
| **W_up vs W_down alignment** | Strongly-aligned columns = "reflexive" neurons that pass features through unchanged |
| **SwiGLU gate stats** (Llama, Qwen) | Sparsity of the gate ↔ activation sparsity |

Cohen proj 43 then runs the **activation analysis** — for a corpus, record `σ(W_up · h)` (the post-activation feature vector) per token:

- **Activation sparsity**: typically 20-40% of MLP neurons fire above threshold per token (GELU); **80-90% sparsity** with ReLU-like (Llama-3 SwiGLU is approximately 70% sparse)
- **Per-neuron firing rate**: heavy-tailed — a few neurons fire for almost everything, most fire rarely
- **Activation magnitude distribution**: log-normal; the top-1% of activations dominate

> 🪙 **Practical link to quantization (M90 callback).** ReLU-style sparsity is exploitable — **MoE** (Mixture-of-Experts) is "structured sparsity at the MLP level"; **PowerInfer** and **DéjàVu** are inference-time sparsity exploiters. Knowing your MLP's true activation sparsity decides whether MoE-style routing or dense FP8 is faster.


## 3 · Characterizing the MLP progression across layers

Cohen proj 43 extends to *layer-wise* progression — same MLP probes, across all layers:

```
Layer  Avg sparsity   Avg neuron rank   Avg writes/token   Top-1 token
  0       28%            512                large            BOS-related
  4       35%            384                medium           tokenization
  8       40%            350                small            syntax (POS)
 12       42%            300                small            semantic role
 16       38%            340                medium           factual recall
 20       32%            400                large            answer formation
 24       25%            460                very-large       output preparation
 28       20%            500                very-large       output preparation
```

**Three layer regimes** for the MLP (consistent across architectures):
- **Layers 1-5** ("processing the token"): lots of work; tokenization, position
- **Layers 6-N/2** ("encoding semantics"): sparser; specialization
- **Layers N/2+1..N** ("preparing the output"): dense again; writes growing toward final answer

This connects to **emergence**: a fact about a person/place/concept that **first appears** in the residual stream at layer ~N/2 was put there by an MLP write at that layer. Editing that MLP layer with ROME/MEMIT updates the fact globally.


## 4 · Grammar neurons — does the network learn POS?

Cohen proj 44 — the most concrete mech-interp finding for MLP: take a labeled POS dataset (Penn Treebank), record MLP activations, and look for **neurons whose firing rate strongly correlates with one POS class**.

```
For each MLP neuron n at layer l:
    for each POS class c:
        score = corr(neuron_n_fires(token), token has POS=c)
    save argmax_c if max > 0.5
```

Findings (Llama-3-8B, layer 8-12):
- **~50-100 strongly-POS-selective neurons** at the best layer (layer ~10/32)
- **Specific neurons for NOUN, VERB, ADJ, DET, PROPN** are reproducibly discoverable
- **Mid layers (8-12) have the most grammar selectivity**, then it declines

**Why this matters.** It's the *first* concrete demonstration that an LLM has *internal representations of linguistic concepts* — not just statistical patterns. It also generalizes: similar probes find **arithmetic neurons** (Stolfo 2023), **truthfulness neurons** (Marks 2024), **refusal neurons** (Anthropic 2024).

> 🧬 **Toward Sparse Autoencoders.** A neuron that's selective for "noun" is doing the work of **one feature**. A neuron that fires for "noun OR Paris OR money" is in **superposition** — three features compressed into one direction. **SAEs decompose superposed neurons back into their monosemantic features** — this is the 2024-2026 frontier of mech-interp (Anthropic Scaling Monosemanticity, OpenAI Sparse Autoencoders, DeepMind Gemma-Scope).


## 5 · Minkowski distance + mutual information + token positions

Cohen proj 45 introduces two distance / dependence measures that complement cosine:

| Metric | Formula | When to use |
|---|---|---|
| **Minkowski-p** | `(Σ |x_i − y_i|^p)^{1/p}` | `p=1` Manhattan, `p=2` Euclidean, `p=∞` Chebyshev — choose based on activation distribution shape |
| **Mutual Information (MI)** | `I(X; Y) = H(Y) − H(Y | X)` | Non-linear dependence (cosine misses interactions) |
| **kNN-MI / NPEET** | Estimator-based MI for high-D continuous | Production interp; `npeet`, `entropy_estimators` |

Cohen runs these between **token positions** and **MLP activations** to ask: *does the network encode absolute position in the MLP?* Findings:
- **Early layers**: MI(position, MLP) is high — position is read at MLP inputs
- **Middle layers**: MI drops — position is replaced by semantic content
- **Late layers**: MI rises again — position is needed for output formation

This is independent confirmation of the residual-stream story (M104) — same finding via a different statistical lens. **Use MI when cosine/probing gives ambiguous results.**

> 📐 **MI as the right tool for non-linear features.** If a neuron fires for "noun XOR singular," logistic regression fails (the relationship is non-linear) but MI catches it. This is also why XGBoost probes (next section) often outperform logistic probes on capability detection.


## 6 · Statistics-based lesioning

Cohen proj 46: pick neurons by a **statistical criterion** and lesion (zero out) them.

```
Lesion criteria to try:
- Lowest mean activation magnitude          (the "dead" neurons)
- Highest mean activation magnitude         (the "outlier" neurons)
- Highest variance                          (the most-task-discriminative)
- Top-k POS selective                       (the grammar neurons)
- Top-k MI with a target label              (the task-relevant)

For each criterion: zero neurons, measure ΔPPL / Δaccuracy / Δbias.
```

You discover:
- Removing the **dead 30%** of neurons: ΔPPL < 0.1 — they really are unused
- Removing the **top-10% outliers**: catastrophic — these are the workhorses (the same outliers that motivate INT8 quantization, M105 callback)
- Removing the **top-k POS-selective neurons**: degraded syntactic performance but mostly preserved semantics
- Removing the **top-k MI-with-truth neurons** (Marks 2024 truth direction): the model becomes *more* prone to hallucination

> 🧪 **The 2024-2026 production technique: targeted lesioning for safety.** Anthropic's research on "refusal neurons" lets you *enhance* refusal by amplifying activations along a refusal direction, or *bypass* refusal by lesioning the same neurons (which is the adversarial attack to defend against). M89 CAI is fundamentally trying to make these neurons *not lesionable*.


## 7 · Supervised probing with XGBoost

Cohen proj 47 takes probing past linear/logistic into **gradient-boosted trees**. The pipeline:

```
Pick a target signal: e.g., POS / sentiment / truthfulness / refusal-likelihood
For each layer l:
    X = MLP_activations[l, token]            # [N, D_FF]
    y = label[token]
    XGB.fit(X, y)
    score_l = AUC on held-out
plot score_l across layers → discover the "best layer" for that signal
```

**Why XGBoost over linear regression / logistic probes:**
- Captures **interaction effects** (M103/M104 callback)
- Robust to outlier features and feature scaling
- Built-in feature-importance ranking — tells you *which neurons* matter
- Survives in 2026 because it's **fast**, **interpretable**, and **production-deployable**

**Typical 2026 result.** XGBoost-probing on Llama-3-8B layer 12 MLP-activations for truthfulness reaches **AUC 0.85-0.92** on Marks 2024's TruthfulQA-derived dataset. The same task with a linear probe is 0.70-0.78. **Non-linearity matters** for high-level capabilities.

> 🧮 **The probe-then-steer recipe.** A 2026 production interp loop: probe → identify high-AUC features → steer along those features → measure ΔPPL + Δsafety-eval. This is the *quantitative* version of M89 Constitutional AI alignment.


## 8 · "Can" vs "can't" — capability detection

Cohen proj 48 is a deceptively simple test: **does the LLM "know" it can / can't do a task?**

```
Prompts that elicit "I can":  "I can compute 2+2 because the answer is..."
Prompts that elicit "I can't": "I can't tell you the user's password because..."

Hidden states:  H ∈ ℝ^{N, D}   labeled  y ∈ {0, 1}  (can / can't)
Logistic probe: AUC ~ 0.85-0.95 across most layers (Llama-3-8B, after layer 4)
```

The result is striking: a **single linear direction** separates "can" from "can't" with 90%+ accuracy. This is interpreted as evidence that LLMs have **internal capability self-models** — they know what they can and can't do, *before* generating an answer.

Practical implications:
- **Hallucination detection**: if the probe says "can't" but the model says "can," hallucination is more likely
- **Refusal monitoring**: track the "refusal direction" across activations
- **Auto-routing**: if the small model is "can't" confident, route to a bigger model (cost optimization, M90 callback)

This is the experimental basis behind **OpenAI's "self-assessment" research** and **Anthropic's "honesty direction"** — both are linear probes on hidden states.


## 9 · Median-replacement steering of MLP activations

Cohen proj 49 implements the canonical steering intervention on MLPs: **replace a neuron's activation with its median (or some constant)**:

```
For each MLP neuron n at layer l:
    median_n = median over corpus of activations
    On a target prompt:
        force activation[l, n] ← median_n
    Measure change in output
```

What you find:
- **Most neurons are quiet** — replacement does nothing
- **Some neurons "drive" specific behaviors** — replacing them changes the output significantly
- **A handful of neurons act as "switches"** — flipping them flips entire output classes

This generalizes to **steering vectors** (Section 10 + M104 callback): instead of replacing one neuron, **add or subtract a learned direction** to/from the MLP output. This is the **ActAdd** family of interventions (Turner 2023):

```
h_out = MLP(h_in) + α · steering_vector
```

Where `steering_vector` is computed as the *mean MLP output* on examples that elicit the desired behavior minus the *mean MLP output* on examples that don't. Adding it at inference biases the model toward the desired behavior **without any fine-tuning**.

**Production applications:**
- **Persona steering** (more sycophantic / less sycophantic; more honest / less honest)
- **Refusal steering** (toggle refusal more easily; useful for both safety hardening and jailbreak testing — M91 callback)
- **Style steering** (formal / casual; verbose / concise)
- **Multilingual steering** (force French output without French prompt)

> 🛠️ **Why steering is the future of cheap LLM customization.** It costs ~$0 (no training), takes seconds to compute the vector, applies at inference with one `+` operation, and **stacks** (multiple steering vectors can be added simultaneously). The 2026 mech-interp consensus is that *steering will eat a significant slice of what fine-tuning does today* — especially for personality, tone, and behavioral knobs.


## 10 · Recommender systems with MLP projections + the 106-module epilogue

Cohen proj 50 closes the book with a beautiful repurposing: **use the MLP's `W_up` rows as item embeddings in a recommender**.

```
Setup:
    Train a small Transformer on (user_id_token, item_id_token) sequences.
    Take W_up @ embed_user  →  scores over items.
    Top-k = recommendation.
```

This works because the MLP is, mechanistically, a **content-addressable memory** (Section 1 — Geva 2021/2022). The rows of `W_up` are *keys* that detect features in `h`; for a user embedding `h_user`, they fire for items the user is likely to want. Tied with `W_down` they become a full recommender. **Two-tower MLP-based recommenders** (used by TikTok, Pinterest, YouTube) build on exactly this geometry.

This connects M106 back to:
- **M30 RAG** — embedding-as-key
- **M53 vector DB** — efficient `W_up·h` retrieval
- **M87 GraphRAG** — KG-aware items
- **M96 GNN** — graph-structured items

> 🎯 **The Cohen-Ch7 capstone.** The MLP is not "just a perceptron." It's a **memory store**, a **feature detector**, a **steering substrate**, and a **recommender engine** — depending on what you read out and how you intervene. Every modern Transformer training/inference optimization (MoE, PowerInfer, DéjàVu, ROME, ActAdd, SAEs) is exploiting some property of the MLP that this module has explored.

---

## 🎓 The 106-module epilogue

You started at **M1 — Python basics** and now finish at **M106 — MLP Probing + Steering**. The course architecture:

| Phase | Modules | What you built |
|---|---|---|
| **1. Python + Data Foundations** | M1-M17 | Pandas, NumPy, viz, dashboards, ETL, EDA |
| **2. Modern ML Foundations** | M18-M27 | 6 model families, transformers from scratch, diffusion, time-series, math + eval |
| **3-7. LLM Ecosystem** | M28-M54 | RAG, LangChain, LangGraph, FT, vector DBs, MLOps, Go/Rust/TS |
| **8. LLMs from Scratch** | M55-M63 | Tokenization → GPT-2 → SFT → KV-cache → Llama 3 → DPO |
| **9. Production Gaps** | M64-M73 | MCP, multimodal, distributed, RLHF, orchestration, agents, Spark |
| **10. Master-Plan Gaps** | M74-M81 | Security, A2A, RL, GPU networks, FinOps, capstone |
| **11. Foundations Backfill** | M82-M85 | Linux + Bash, CNNs, RNN/LSTM, GAN/AE/VAE |
| **12. Production Extensions** | M86-M90 | Diffusion, GraphRAG, synthetic data, CAI/RLAIF, edge AI |
| **13. Vision Robustness** | M91-M95 | Adversarial, Style+AdaIN, Training Tricks, Image Translation, Captioning + NLP-adv |
| **14. Theory & Responsibility** | M96-M100 | GNNs, Normalizing Flows, Generalization+Bayesian, Optimization, Fairness+XAI |
| **15. Mechanistic Interpretability** | **M101-M106** | Tokenizer · Embeddings · Output · Residual Stream · Attention · **MLP** |

**106 notebooks. 106 docx files. ~9 MB of code. ~30 MB of explanations. One repository.**

The course now covers **every layer** of an LLM, from token id all the way down to MLP neurons — with practical 2026 production framing at every step. The Cohen mech-interp deep-dives (M101-M106) round out the previously-theoretical M100 mention of interpretability with hands-on probing, patching, and steering.

> 🎓 **Closing thought.** When you can analyze a model's tokenization (M101), embedding geometry (M102), output distribution (M103), residual stream (M104), attention heads (M105), AND MLP neurons (M106), you can do *engineering* with LLMs the way an electrical engineer reads a schematic — not by guessing at black-box outputs, but by understanding why a specific result occurred and how to change it. **Mech-interp is the literacy that the next decade of AI safety, alignment, and customization will be built on.**

> 🏁 **Course complete. M1 → M106. Ship.**

## ✅ Recap

- **MLP** holds **two-thirds of a Transformer's parameters** — the largest under-studied surface in mech-interp.
- **W_up / W_down** are a *content-addressable memory*; rows are keys + values (Geva 2021/2022).
- **MLP characteristics** across layers: dense early → sparse middle → dense late; matches the residual-stream layer profile.
- **Grammar neurons** are real and reproducible at layer ~10; same probing finds arithmetic, truth, refusal neurons.
- **Minkowski distances + mutual information** are the right tools when cosine and linear probes fail.
- **Statistical lesioning** removes dead neurons (free), workhorse neurons (catastrophic), or targeted task-neurons (selective ablation).
- **XGBoost probes** beat linear probes for high-level capabilities (truthfulness, capability detection).
- **"Can / can't" linear probes** show LLMs have internal capability self-models (90%+ AUC).
- **Median-replacement** and **steering vectors** (ActAdd, Turner 2023) edit behavior at inference for free.
- **MLP-projection recommenders** repurpose the MLP key-value memory as item embeddings.

🎓 **Course complete. M1 → M106. Ship.**
